In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))  # for ms_plot_utils
sys.path.insert(0, str(PROJECT_ROOT / "ciceroscm-surrogate" / "notebooks"))  # for plot_utils, plot_comparison

from plot_utils import plot_training_time_and_speedup, plot_reward, _extract_lever_table, _extract_adaptation_table, plot_lever_consistency, plot_lever_consistency_mean
from plot_comparison import plot_effort_comparison, plot_reward_comparison
from ms_plot_utils import plot_multiple_reward, plot_lever_consistency_dual, plot_reward_components, _extract_promises_table, _extract_acceptance_rate_table, plot_negotiation

# 1: Climate damage

## 1. Homogenous agents

### 1.1 Load data

In [ ]:
results_path = "../data/20260507_140427/marl_results/bilateral_agreements/high_damage/nonbinding/20260601_224119_20260507_145205_gru_256_1_homogenous_agents_nb/ms_marl_experiment_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

training_time = pd.DataFrame.from_dict(results["training_time_stats"], orient="index", columns=["value"])
greedy_reward = pd.DataFrame.from_dict(results["greedy_reward"], orient="index")
train_reward = pd.DataFrame.from_dict(results["train_reward"], orient="index")
greedy_policy = results["greedy_policy"]

energy      = _extract_lever_table(greedy_policy, "energy")
methane      = _extract_lever_table(greedy_policy, "methane")
agriculture  = _extract_lever_table(greedy_policy, "agriculture")
adaptation   = _extract_adaptation_table(greedy_policy)

### 1.2 Reward

#### 1.2.1 Training reward

In [ ]:
plot_reward(train_reward, steps_max=4_000_000,cmap="ocean", y_min = -56, y_max =-52, series_name="Country", ylabel="Training reward")

#### 1.2.2 Greedy reward

In [ ]:
plot_reward(greedy_reward, steps_max=4_000_000,cmap="ocean", y_min = -70, y_max =-40, series_name="Country", ylabel="Greedy reward")

### 1.3 Policy levers

In [ ]:
lever_series = {
    "energy": (energy),
    "methane": (methane),
    "agriculture": (agriculture),
    "adaptation": (adaptation)
}
plot_lever_consistency(lever_series, agent_mask=[1,1,1,1,1,1,1,1,1,1], action_mask=[1,1,1,1], y_min=-0.01, y_max=1.01, steps_max=1_750_000, series_name="Country")

### 1.4 Temperature trajectory

In [ ]:
temp_traj = results["temperature_trajectory"]
steps = sorted(temp_traj.keys(), key=int)
final_temp = temp_traj[steps[-1]]

years = list(range(2016, 2016 + len(final_temp)))  # 2016–2065 (35 policy + 15 terminal rollout)

max_effort_path = "../data/20260507_140427/marl_results/baseline/max_effort_results.json"
min_effort_path = "../data/20260507_140427/marl_results/baseline/min_effort_results.json"

with open(max_effort_path) as f:
    max_effort_temp = json.load(f)["temperature_trajectory"]
with open(min_effort_path) as f:
    min_effort_temp = json.load(f)["temperature_trajectory"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(years, min_effort_temp, color="#808080", linewidth=2.2, linestyle=":", label="No policy")
ax.plot(years, final_temp,      color="#1f78b4", linewidth=2.2,                  label="Learned policy")
ax.plot(years, max_effort_temp, color="#e31a1c", linewidth=2.2, linestyle=":",  label="Max effort")
ax.axvline(2085, color="#999999", linestyle="--", linewidth=1.0, label="Policy end")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Temperature anomaly (K)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(frameon=True, fontsize=12)
plt.tight_layout()
plt.show()

## 2. Heterogenous agents

### 2.1 Load data

In [ ]:
results_path = "../data/20260307_122142/marl_results/heterogenous_experiments/heterogenous_shares/20260502_210301_20260307_174513_gru_128_1_heterogenous_agents/marl_experiment_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

training_time = pd.DataFrame.from_dict(results["training_time_stats"], orient="index", columns=["value"])
greedy_reward = pd.DataFrame.from_dict(results["greedy_reward"], orient="index")
train_reward = pd.DataFrame.from_dict(results["train_reward"], orient="index")
greedy_policy = results["greedy_policy"]

energy      = _extract_lever_table(greedy_policy, "energy")
methane      = _extract_lever_table(greedy_policy, "methane")
agriculture  = _extract_lever_table(greedy_policy, "agriculture")
adaptation   = _extract_adaptation_table(greedy_policy)

### 2.2 Reward

#### 2.2.1 Training reward

In [ ]:
plot_reward(train_reward, steps_max=750_000,cmap="ocean", y_min = -4, y_max =-1.5, series_name="Country", ylabel="Training reward")

#### 2.2.2 Greedy reward

In [ ]:
plot_reward(greedy_reward, steps_max=750_000,cmap="ocean", y_min = -4, y_max =-1.5, series_name="Country", ylabel="Greedy reward")

### 2.3 Policy levers

In [ ]:
lever_series = {
    "energy": (energy),
    "methane": (methane),
    "agriculture": (agriculture),
    "adaptation": (adaptation)
}
plot_lever_consistency(lever_series, agent_mask=[1,1,1,1,1,1,1,1,1,1], action_mask=[1,1,1,1], y_min=-0.01, y_max=1.01, steps_max=750_000, series_name="Country")

### 2.4 Temperature trajectory

In [ ]:
temp_traj = results["temperature_trajectory"]
steps = sorted(temp_traj.keys(), key=int)
final_temp = temp_traj[steps[-1]]

years = list(range(2016, 2016 + len(final_temp)))  # 2016–2065 (35 policy + 15 terminal rollout)

max_effort_path = "../data/20260307_122142/marl_results/homogenous_experiments/20260429_143821_20260307_174513_gru_128_1_homogenous_agents/max_effort_results.json"
min_effort_path = "../data/20260307_122142/marl_results/homogenous_experiments/20260502_210121_20260307_174513_gru_128_1_heterogenous_agents/min_effort_results.json"

with open(max_effort_path) as f:
    max_effort_temp = json.load(f)["temperature_trajectory"]
with open(min_effort_path) as f:
    min_effort_temp = json.load(f)["temperature_trajectory"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(years, min_effort_temp, color="#808080", linewidth=2.2, linestyle=":", label="No policy")
ax.plot(years, final_temp,      color="#1f78b4", linewidth=2.2,                  label="Learned policy")
ax.plot(years, max_effort_temp, color="#e31a1c", linewidth=2.2, linestyle="--",  label="Max effort")
ax.axvline(2050, color="#999999", linestyle="--", linewidth=1.0, label="Policy end")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Temperature anomaly (K)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(frameon=True, fontsize=12)
plt.tight_layout()
plt.show()


# 2: Bilateral agreements

## 3. Nonbinding agreements

### 3.1 Load data

In [ ]:
results_path = "../data/20260507_140427/marl_results/bilateral_agreements/medium_damage/nonbinding/20260531_225158_20260507_145205_gru_256_1_homogenous_agents_nb/ms_marl_experiment_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

training_time = pd.DataFrame.from_dict(results["training_time_stats"], orient="index", columns=["value"])
greedy_reward = pd.DataFrame.from_dict(results["greedy_reward"], orient="index")
train_reward = pd.DataFrame.from_dict(results["train_reward"], orient="index")
greedy_policy = results["greedy_policy"]

energy      = _extract_lever_table(greedy_policy, "energy")
methane      = _extract_lever_table(greedy_policy, "methane")
agriculture  = _extract_lever_table(greedy_policy, "agriculture")
adaptation   = _extract_adaptation_table(greedy_policy)

### 3.2 Reward

#### 3.2.1 Training reward

In [ ]:
plot_reward(train_reward, steps_max=3_050_000,cmap="ocean", y_min = -13.5, y_max =-12, series_name="Country", ylabel="Training reward")

#### 3.2.2 Greedy reward

In [ ]:
plot_reward(greedy_reward, steps_max=3_050_000,cmap="ocean", y_min = -35, y_max =-21, series_name="Country", ylabel="Greedy reward")

#### 3.2.3 Comparing rewards

In [ ]:
ss_path = "../data/20260507_140427/marl_results/bilateral_agreements/medium_damage/nonbinding/20260601_021730_20260507_145205_gru_256_1_homogenous_agents_nb/ms_marl_experiment_results.json"

with open(ss_path, "r") as f:
    ss = json.load(f)

ss_greedy_reward = pd.DataFrame.from_dict(ss["greedy_reward"], orient="index")
ss_greedy_policy = ss["greedy_policy"]

ss_energy      = _extract_lever_table(ss_greedy_policy, "energy")
ss_methane      = _extract_lever_table(ss_greedy_policy, "methane")
ss_agriculture  = _extract_lever_table(ss_greedy_policy, "agriculture")
ss_adaptation   = _extract_adaptation_table(ss_greedy_policy)

ss_lever_series = {
    "energy": (ss_energy),
    "methane": (ss_methane),
    "agriculture": (ss_agriculture),
    "adaptation": (ss_adaptation)
}

In [ ]:
plot_multiple_reward(series=[("SS", ss_greedy_reward), ("MS", greedy_reward)], dual_xaxis=True, y_min=-28, y_max=-21, ylabel="Greedy reward")

### 3.3 Policy levers

#### 3.3.1 Levers

In [ ]:
lever_series = {
    "energy": (energy),
    "methane": (methane),
    "agriculture": (agriculture),
    "adaptation": (adaptation)
}
plot_lever_consistency(lever_series, agent_mask=[1,1,1,1,1,1,1,1,1,1], action_mask=[1,1,1,1], y_min=-0.01, y_max=1.01, steps_max=4_050_000, series_name="Country")

#### 3.3.2 Comparing levers

In [ ]:
plot_lever_consistency_dual(
    {lever: (ss_lever_series[lever], lever_series[lever]) for lever in ss_lever_series},
    [1,1,1,1,1,1,1,1,1,1],
    [1,1,1,1],
    y_min=-0.01,
    y_max=1.01,
)

### 3.4 Temperature trajectory

In [ ]:
ss_traj = ss["temperature_trajectory"]
ss_steps = sorted(ss_traj.keys(), key=int)
ss_final_temp = ss_traj[ss_steps[-1]]

temp_traj = results["temperature_trajectory"]
steps = sorted(temp_traj.keys(), key=int)
final_temp = temp_traj[steps[-1]]

years = list(range(2016, 2016 + len(final_temp)))  # 2016–2065 (35 policy + 15 terminal rollout)

max_effort_path = "../data/20260507_140427/marl_results/baseline/max_effort_results.json"
min_effort_path = "../data/20260507_140427/marl_results/baseline/min_effort_results.json"

with open(max_effort_path) as f:
    max_effort_temp = json.load(f)["temperature_trajectory"]
with open(min_effort_path) as f:
    min_effort_temp = json.load(f)["temperature_trajectory"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(years, min_effort_temp, color="#808080", linewidth=2.2, linestyle=":", label="No policy")
ax.plot(years, ss_final_temp,   color="#808080", linewidth=2.2, label="SS policy")
ax.plot(years, final_temp,      color="#1f78b4", linewidth=2.2, label="Learned policy")
ax.plot(years, max_effort_temp, color="#e31a1c", linewidth=2.2, linestyle="--",  label="Max effort")
ax.axvline(2065, color="#999999", linestyle="--", linewidth=1.0, label="Policy end")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Temperature anomaly (K)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(frameon=True, fontsize=12)
plt.tight_layout()
plt.show()


## 4. Noncompliance penalty

### 4.1 Load data

In [ ]:
results_path = "../data/20260507_140427/marl_results/bilateral_agreements/medium_damage/nonbinding/20260601_021730_20260507_145205_gru_256_1_homogenous_agents_nb/ms_marl_experiment_results.json"

with open(results_path, "r") as f:
    results = json.load(f)

training_time = pd.DataFrame.from_dict(results["training_time_stats"], orient="index", columns=["value"])
greedy_reward = pd.DataFrame.from_dict(results["greedy_reward"], orient="index")
train_reward = pd.DataFrame.from_dict(results["train_reward"], orient="index")
climate_reward = pd.DataFrame.from_dict(results["climate_reward"], orient="index")
greedy_policy = results["greedy_policy"]

energy      = _extract_lever_table(greedy_policy, "energy")
methane      = _extract_lever_table(greedy_policy, "methane")
agriculture  = _extract_lever_table(greedy_policy, "agriculture")
adaptation   = _extract_adaptation_table(greedy_policy)

### 4.2 Reward

#### 4.2.1 Training reward

In [ ]:
plot_reward(train_reward, steps_max=3_050_000,cmap="ocean", y_min = -28, y_max =-22, series_name="Country", ylabel="Training reward")

#### 4.2.2 Greedy reward

In [ ]:
plot_reward(greedy_reward, steps_max=3_050_000,cmap="ocean", y_min = -32, y_max =-21, series_name="Country", ylabel="Greedy reward")

#### 4.2.3 Climate reward

In [ ]:
plot_reward(climate_reward, steps_max=4_050_000,cmap="ocean", y_min = -28, y_max =-21, series_name="Country", ylabel="Climate reward")

#### 4.2.4 Greedy reward components

In [ ]:
plot_reward_components(results,
    component_keys=[
        ("noncompliance_cost", "Noncompliance penalty")
    ],
)

#### 4.2.5 Comparing rewards

In [ ]:
plot_multiple_reward(series=[("SS", ss_greedy_reward), ("MS", climate_reward)], dual_xaxis=True, y_min=-28, y_max=-21, ylabel="Greedy (climate) reward")

### 4.3 Policy levers

In [ ]:
lever_series = {
    "energy": (energy),
    "methane": (methane),
    "agriculture": (agriculture),
    "adaptation": (adaptation)
}
plot_lever_consistency(lever_series, agent_mask=[1,1,1,1,1,1,1,1,1,1], action_mask=[1,1,1,1], y_min=-0.01, y_max=1.01, steps_max=4_050_000, series_name="Country")

### 4.4 Temperature trajectory

In [ ]:
nb_path = "../data/20260507_140427/marl_results/bilateral_agreements/medium_damage/nonbinding/20260518_174236_20260507_145205_gru_256_1_homogenous_agents_nb/ms_marl_experiment_results.json"

with open(nb_path, "r") as f:
    nb = json.load(f)

nb_traj = nb["temperature_trajectory"]
nb_steps = sorted(nb_traj.keys(), key=int)
nb_final_temp = nb_traj[nb_steps[-1]]

temp_traj = results["temperature_trajectory"]
steps = sorted(temp_traj.keys(), key=int)
final_temp = temp_traj[steps[-1]]

years = list(range(2016, 2016 + len(final_temp)))  # 2016–2065 (35 policy + 15 terminal rollout)

max_effort_path = "../data/20260507_140427/marl_results/baseline/max_effort_results.json"
min_effort_path = "../data/20260507_140427/marl_results/baseline/min_effort_results.json"

with open(max_effort_path) as f:
    max_effort_temp = json.load(f)["temperature_trajectory"]
with open(min_effort_path) as f:
    min_effort_temp = json.load(f)["temperature_trajectory"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(years, min_effort_temp, color="#808080", linewidth=2.2, linestyle=":", label="No policy")
#ax.plot(years, ss_final_temp,   color="#808080", linewidth=2.2, linestyle="--", label="SS policy")
ax.plot(years, nb_final_temp,   color="#35C825", linewidth=2.2, label="Nonbinding")
ax.plot(years, final_temp,      color="#1f78b4", linewidth=2.2, label="Noncompliance")
ax.plot(years, max_effort_temp, color="#e31a1c", linewidth=2.2, linestyle="--",  label="Max effort")
ax.axvline(2050, color="#999999", linestyle="--", linewidth=1.0, label="Policy end")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Temperature anomaly (K)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(frameon=True, fontsize=12)
plt.tight_layout()
plt.show()
